# SyMTRS Bidirectional Inference (5 images)

This notebook uses training-style paired crops (`tile_size=512`, same crop box for day and night), then runs:
- CycleGAN Day-to-night (`G_A`)
- CycleGAN Night-to-day (`G_B`)
- pix2pix Day-to-night
- pix2pix Night-to-day

Final plot order:
`GT: day | GT: night | CycleGAN Day-to-night | CycleGAN Night-to-day | pix2pix Day-to-night | pix2pix Night-to-day`


In [ ]:
from pathlib import Path
from PIL import Image
import random

files = [
    'RS.68400.png',
    'RS.100200.png',
    'RS.1000200.png',
    'RS.675600.png',
    'RS.1206000.png',
]

repo = Path('.').resolve()
src_day = Path('/home/data/SyMTRS/hr')
src_night = Path('/home/data/SyMTRS/night')
crop_day = repo / 'datasets/symtrs_subset5_crops/day'
crop_night = repo / 'datasets/symtrs_subset5_crops/night'
crop_day.mkdir(parents=True, exist_ok=True)
crop_night.mkdir(parents=True, exist_ok=True)

tile_size = 512
seed = 42
rng = random.Random(seed)

for f in files:
    day = Image.open(src_day / f).convert('RGB')
    night = Image.open(src_night / f).convert('RGB')
    w, h = day.size

    if w < tile_size or h < tile_size:
        day_c, night_c = day, night
        left, top = 0, 0
    else:
        left = rng.randint(0, w - tile_size)
        top = rng.randint(0, h - tile_size)
        box = (left, top, left + tile_size, top + tile_size)
        day_c = day.crop(box)
        night_c = night.crop(box)

    day_c.save(crop_day / f)
    night_c.save(crop_night / f)
    print(f'{f}: left={left}, top={top}, size={day_c.size}')


In [ ]:
import subprocess

# CycleGAN Day-to-night (G_A)
subprocess.run([
    'python', 'test.py',
    '--dataroot', 'datasets/symtrs_subset5_crops/day',
    '--name', 'symtrs_cyclegan_t512_e200_s42',
    '--model', 'test',
    '--model_suffix', '_A',
    '--netG', 'resnet_9blocks',
    '--no_dropout',
    '--results_dir', 'results/symtrs_subset5_crops_bidir',
    '--num_test', '5',
    '--epoch', 'latest',
    '--phase', 'd2n',
    '--preprocess', 'none',
    '--eval',
], check=True)

# CycleGAN Night-to-day (G_B)
subprocess.run([
    'python', 'test.py',
    '--dataroot', 'datasets/symtrs_subset5_crops/night',
    '--name', 'symtrs_cyclegan_t512_e200_s42',
    '--model', 'test',
    '--model_suffix', '_B',
    '--netG', 'resnet_9blocks',
    '--no_dropout',
    '--results_dir', 'results/symtrs_subset5_crops_bidir',
    '--num_test', '5',
    '--epoch', 'latest',
    '--phase', 'n2d',
    '--preprocess', 'none',
    '--eval',
], check=True)

# pix2pix Day-to-night
subprocess.run([
    'python', 'test.py',
    '--dataroot', 'datasets/symtrs_subset5_crops/day',
    '--name', 'symtrs_pix2pix_t512_e200_s42',
    '--model', 'test',
    '--netG', 'unet_256',
    '--norm', 'batch',
    '--results_dir', 'results/symtrs_subset5_crops_bidir',
    '--num_test', '5',
    '--epoch', 'latest',
    '--phase', 'd2n',
    '--preprocess', 'none',
    '--eval',
], check=True)

# pix2pix Night-to-day (same checkpoint, night input)
subprocess.run([
    'python', 'test.py',
    '--dataroot', 'datasets/symtrs_subset5_crops/night',
    '--name', 'symtrs_pix2pix_t512_e200_s42',
    '--model', 'test',
    '--netG', 'unet_256',
    '--norm', 'batch',
    '--direction', 'BtoA',
    '--results_dir', 'results/symtrs_subset5_crops_bidir',
    '--num_test', '5',
    '--epoch', 'latest',
    '--phase', 'n2d',
    '--preprocess', 'none',
    '--eval',
], check=True)

print('All four inference runs finished.')


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

base = Path('results/symtrs_subset5_crops_bidir')
cyc_d2n = base / 'symtrs_cyclegan_t512_e200_s42/d2n_latest/images'
cyc_n2d = base / 'symtrs_cyclegan_t512_e200_s42/n2d_latest/images'
pix_d2n = base / 'symtrs_pix2pix_t512_e200_s42/d2n_latest/images'
pix_n2d = base / 'symtrs_pix2pix_t512_e200_s42/n2d_latest/images'
gt_day = Path('datasets/symtrs_subset5_crops/day')
gt_night = Path('datasets/symtrs_subset5_crops/night')

fig, axes = plt.subplots(len(files), 6, figsize=(24, 4 * len(files)))

for i, f in enumerate(files):
    stem = f.rsplit('.', 1)[0]
    images = [
        Image.open(gt_day / f),
        Image.open(gt_night / f),
        Image.open(cyc_d2n / f'{stem}_fake.png'),
        Image.open(cyc_n2d / f'{stem}_fake.png'),
        Image.open(pix_d2n / f'{stem}_fake.png'),
        Image.open(pix_n2d / f'{stem}_fake.png'),
    ]

    titles = [
        'GT: day',
        'GT: night',
        'CycleGAN Day-to-night',
        'CycleGAN Night-to-day',
        'pix2pix Day-to-night',
        'pix2pix Night-to-day',
    ]

    for j in range(6):
        axes[i, j].imshow(images[j])
        axes[i, j].set_title(titles[j] if i == 0 else '')
        axes[i, j].axis('off')

plt.tight_layout()
out_plot = base / 'comparison_6cols_bidir.png'
plt.savefig(out_plot, dpi=150)
print('Saved plot:', out_plot)
plt.show()
